# Fox Audio Detection — Project Setup

## Step 1: Create Project Directory Structure & Conda Environment

This cell creates the full `fox_detection/` project tree and verifies it.

In [19]:
import os

# Project root
PROJECT_ROOT = os.path.join(os.getcwd(), "fox_detection")

# Define directory structure
directories = [
    "data/raw/fox",
    "data/raw/nonfox",
    "data/clips/fox",
    "data/clips/nonfox",
    "data/features",
    "data/spectrograms/fox",
    "data/spectrograms/nonfox",
    "models/baseline",
    "models/cnn",
    "notebooks",
    "src",
]

# Create directories
for d in directories:
    path = os.path.join(PROJECT_ROOT, d)
    os.makedirs(path, exist_ok=True)
    print(f"✓ {d}/")

print(f"\nProject root: {PROJECT_ROOT}")

✓ data/raw/fox/
✓ data/raw/nonfox/
✓ data/clips/fox/
✓ data/clips/nonfox/
✓ data/features/
✓ data/spectrograms/fox/
✓ data/spectrograms/nonfox/
✓ models/baseline/
✓ models/cnn/
✓ notebooks/
✓ src/

Project root: /Users/dyx/Documents/TECHIN513A/Final/fox_detection


In [20]:
# Verify the complete project structure
for root, dirs, files in os.walk(PROJECT_ROOT):
    # Skip hidden dirs
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    level = root.replace(PROJECT_ROOT, '').count(os.sep)
    indent = ' ' * 2 * level
    basename = os.path.basename(root)
    print(f'{indent}{basename}/')
    subindent = ' ' * 2 * (level + 1)
    for file in sorted(files):
        if not file.startswith('.'):
            print(f'{subindent}{file}')

fox_detection/
  README.md
  environment.yml
  requirements.txt
  models/
    baseline/
    cnn/
  data/
    manifest.csv
    clips/
      nonfox/
      fox/
    features/
    spectrograms/
      nonfox/
      fox/
    raw/
      nonfox/
      fox/
  notebooks/
    01_data_exploration.ipynb
    02_preprocessing.ipynb
    03_baseline_model.ipynb
    04_cnn_model.ipynb
    05_evaluation.ipynb
  src/
    __init__.py
    audio_utils.py
    baseline_model.py
    cnn_model.py
    dataset.py
    demo.py
    evaluate.py
    features.py
    segmentation.py
    train_cnn.py
    __pycache__/
      __init__.cpython-314.pyc
      audio_utils.cpython-314.pyc
      segmentation.cpython-314.pyc


## Environment Configuration

The `environment.yml` file specifies all dependencies for the conda environment:

```yaml
name: fox_detection
channels: [pytorch, conda-forge, defaults]
dependencies:
  python=3.10, librosa, soundfile, numpy, pandas, scikit-learn,
  matplotlib, seaborn, pytorch, torchvision, torchaudio,
  pillow, tqdm, jupyter, gradio (via pip)
```

**To create the environment:**
```bash
cd fox_detection
conda env create -f environment.yml
conda activate fox_detection
```

In [21]:
# Display the environment.yml contents
env_file = os.path.join(PROJECT_ROOT, "environment.yml")
with open(env_file, "r") as f:
    print(f.read())

name: fox_detection
channels:
  - pytorch
  - conda-forge
  - defaults
dependencies:
  - python=3.10
  - librosa
  - soundfile
  - numpy
  - pandas
  - scikit-learn
  - matplotlib
  - seaborn
  - pytorch::pytorch
  - pytorch::torchvision
  - pytorch::torchaudio
  - pillow
  - tqdm
  - jupyter
  - pip
  - pip:
      - gradio



## Source Module Placeholders

All source modules have been created under `fox_detection/src/` with placeholder functions:

| Module | Purpose |
|---|---|
| `audio_utils.py` | Audio loading, resampling, normalization |
| `segmentation.py` | Splitting recordings into 3-second clips |
| `features.py` | MFCC extraction & spectrogram generation |
| `dataset.py` | PyTorch Dataset for spectrogram images |
| `baseline_model.py` | scikit-learn baseline training/inference |
| `cnn_model.py` | CNN architecture (PyTorch) |
| `train_cnn.py` | CNN training loop |
| `evaluate.py` | Metrics computation & visualization |
| `demo.py` | Gradio interactive demo |

Five placeholder notebooks are in `fox_detection/notebooks/` for each project stage.

In [22]:
# List all source files
src_dir = os.path.join(PROJECT_ROOT, "src")
print("Source modules:")
for f in sorted(os.listdir(src_dir)):
    if f.endswith('.py'):
        print(f"  📄 {f}")

# List all notebooks
nb_dir = os.path.join(PROJECT_ROOT, "notebooks")
print("\nNotebooks:")
for f in sorted(os.listdir(nb_dir)):
    if f.endswith('.ipynb'):
        print(f"  📓 {f}")

print("\n✅ Project setup complete!")

Source modules:
  📄 __init__.py
  📄 audio_utils.py
  📄 baseline_model.py
  📄 cnn_model.py
  📄 dataset.py
  📄 demo.py
  📄 evaluate.py
  📄 features.py
  📄 segmentation.py
  📄 train_cnn.py

Notebooks:
  📓 01_data_exploration.ipynb
  📓 02_preprocessing.ipynb
  📓 03_baseline_model.ipynb
  📓 04_cnn_model.ipynb
  📓 05_evaluation.ipynb

✅ Project setup complete!


---

## Step 2: `src/audio_utils.py`

Core audio utilities — loading, normalisation, silence trimming, log-mel spectrograms, spectrogram image export, and MFCC feature extraction.

In [23]:
# Run the audio_utils self-test via its __main__ block
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "src.audio_utils"],
    capture_output=True, text=True,
    cwd=os.path.join(os.getcwd(), "fox_detection") if "fox_detection" not in os.getcwd() else os.getcwd(),
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

=== audio_utils self-test ===

✓ normalise_audio  – peak=1.0000
✓ trim_silence     – 110250 → 68096 samples
✓ log_mel_spec     – shape (128, 130)
✓ save_spec_image  – /var/folders/0m/8x59vmp96730r0vxr_ssqtrh0000gn/T/tmpy__4ujru.png (2555 bytes)
✓ mfcc_features    – shape (80,)

=== all tests passed ===



In [24]:
# Interactive test — import functions and exercise them on a synthetic sine wave
import sys
sys.path.insert(0, os.path.join(os.getcwd(), "fox_detection"))

import numpy as np
from src.audio_utils import (
    normalise_audio,
    trim_silence,
    compute_log_mel_spectrogram,
    save_spectrogram_image,
    compute_mfcc_features,
)

SR = 22050
DURATION = 3.0
t = np.linspace(0, DURATION, int(SR * DURATION), endpoint=False)
sine = (0.5 * np.sin(2 * np.pi * 440 * t)).astype(np.float32)

# 1. Normalise
normed = normalise_audio(sine)
print(f"normalise_audio  → peak = {np.max(np.abs(normed)):.4f}")

# 2. Trim silence (add 1 s padding of zeros)
padded = np.concatenate([np.zeros(SR, dtype=np.float32), sine, np.zeros(SR, dtype=np.float32)])
trimmed = trim_silence(padded, SR)
print(f"trim_silence     → {len(padded):,} → {len(trimmed):,} samples")

# 3. Log-mel spectrogram
spec = compute_log_mel_spectrogram(sine, SR)
print(f"log_mel_spec     → shape {spec.shape}, range [{spec.min():.1f}, {spec.max():.1f}] dB")

# 4. Save spectrogram image
import tempfile
tmp_path = os.path.join(tempfile.gettempdir(), "test_spec.png")
save_spectrogram_image(spec, tmp_path)
print(f"save_spec_image  → {tmp_path} ({os.path.getsize(tmp_path)} bytes)")

# 5. MFCC features
feat = compute_mfcc_features(sine, SR, n_mfcc=40)
print(f"mfcc_features    → shape {feat.shape}  (first 5: {feat[:5]})")

print("\n✅ All audio_utils functions work correctly!")

normalise_audio  → peak = 1.0000
trim_silence     → 110,250 → 68,096 samples
log_mel_spec     → shape (128, 130), range [-80.0, 0.0] dB
save_spec_image  → /var/folders/0m/8x59vmp96730r0vxr_ssqtrh0000gn/T/test_spec.png (2555 bytes)
mfcc_features    → shape (80,)  (first 5: [-475.57126     42.825       28.526556    14.072545    -2.1203027])

✅ All audio_utils functions work correctly!


---

## Step 3: `src/segmentation.py`

Audio segmentation with two strategies and a directory-level pipeline:

| Function | Description |
|---|---|
| `segment_fixed(y, sr, clip_duration, overlap)` | Fixed-length overlapping clips, zero-padded tail |
| `segment_energy(y, sr, …)` | RMS-energy event detection, merging, padding |
| `process_directory(input_dir, output_dir, label, …)` | Load → normalise → segment → save WAVs + DataFrame rows |
| `build_manifest(fox_dir, nonfox_dir, …)` | Process both classes, write combined `manifest.csv` |

**CLI usage:**
```bash
python -m src.segmentation --fox_dir data/raw/fox --nonfox_dir data/raw/nonfox \
                           --out_dir data/clips --manifest data/manifest.csv
```

In [25]:
# Test segmentation functions on a synthetic waveform
from src.segmentation import segment_fixed, segment_energy

# Reuse the sine wave from Step 2: 3 s @ 22050 Hz
print("── segment_fixed ──")
clips_fixed = segment_fixed(sine, SR, clip_duration=1.0, overlap=0.5)
for i, c in enumerate(clips_fixed):
    print(f"  clip {i}: {len(c)} samples ({len(c)/SR:.2f} s)")
print(f"  Total fixed clips: {len(clips_fixed)}")

# Build a test signal for energy-based segmentation:
# 0.5s silence | 1s tone | 0.5s silence | 0.5s tone | 0.5s silence
tone_1 = 0.8 * np.sin(2 * np.pi * 440 * np.arange(int(SR * 1.0)) / SR).astype(np.float32)
tone_2 = 0.8 * np.sin(2 * np.pi * 880 * np.arange(int(SR * 0.5)) / SR).astype(np.float32)
silence = np.zeros(int(SR * 0.5), dtype=np.float32)

test_signal = np.concatenate([silence, tone_1, silence, tone_2, silence])
print(f"\n── segment_energy ──")
print(f"  Test signal: {len(test_signal)} samples ({len(test_signal)/SR:.2f} s)")

clips_energy = segment_energy(test_signal, SR, energy_threshold_db=-30, min_event_duration=0.3)
for i, c in enumerate(clips_energy):
    print(f"  event {i}: {len(c)} samples ({len(c)/SR:.2f} s)")
print(f"  Total energy clips: {len(clips_energy)}")

print("\n✅ Segmentation functions verified!")

── segment_fixed ──
  clip 0: 22050 samples (1.00 s)
  clip 1: 22050 samples (1.00 s)
  clip 2: 22050 samples (1.00 s)
  clip 3: 22050 samples (1.00 s)
  clip 4: 22050 samples (1.00 s)
  clip 5: 22050 samples (1.00 s)
  Total fixed clips: 6

── segment_energy ──
  Test signal: 66150 samples (3.00 s)
  event 0: 28474 samples (1.29 s)
  event 1: 17210 samples (0.78 s)
  Total energy clips: 2

✅ Segmentation functions verified!


---

## Step 4: `src/features.py`

Batch extraction of MFCC features and log-mel spectrogram images from a manifest CSV.

| Function | Description |
|---|---|
| `extract_mfcc_dataset(manifest_csv, …)` | Load each clip → MFCC feature vector → cache as `.npy` → return `(X, y)` arrays |
| `extract_spectrogram_dataset(manifest_csv, …)` | Load each clip → log-mel spectrogram → save as grayscale PNG under `<spec_dir>/<label>/` |

Both functions skip already-processed files for fast re-runs.

**CLI usage:**
```bash
python -m src.features --manifest data/manifest.csv \
                       --feature_dir data/features/ \
                       --spec_dir data/spectrograms/ \
                       --mode both
```

In [ ]:
# Smoke-test features.py using a tiny synthetic manifest
# 1. Create two short WAV clips (fox + nonfox) in a temp directory
import tempfile, shutil, pandas as pd, soundfile as sf
from pathlib import Path

tmp_root = tempfile.mkdtemp(prefix="fox_feat_test_")
clips_dir = os.path.join(tmp_root, "clips")
os.makedirs(os.path.join(clips_dir, "fox"), exist_ok=True)
os.makedirs(os.path.join(clips_dir, "nonfox"), exist_ok=True)

# Synthetic clips: 3 s sine waves at different frequencies
for label, freq, idx in [("fox", 440, 0), ("fox", 500, 1),
                          ("nonfox", 220, 0), ("nonfox", 300, 1)]:
    t_arr = np.linspace(0, 3.0, int(SR * 3), endpoint=False)
    clip = (0.5 * np.sin(2 * np.pi * freq * t_arr)).astype(np.float32)
    sf.write(os.path.join(clips_dir, label, f"clip_{idx:04d}.wav"), clip, SR)

# 2. Build a manifest
rows = []
for label in ("fox", "nonfox"):
    for f in sorted(os.listdir(os.path.join(clips_dir, label))):
        rows.append({
            "file_id": Path(f).stem,
            "source_file": f,
            "clip_path": os.path.join("clips", label, f),
            "label": label,
            "start_sec": 0.0,
            "end_sec": 3.0,
        })
manifest_csv = os.path.join(tmp_root, "manifest.csv")
pd.DataFrame(rows).to_csv(manifest_csv, index=False)
print(f"Test manifest ({len(rows)} rows):")
print(pd.read_csv(manifest_csv).to_string(index=False))

# 3. Run MFCC extraction
from src.features import extract_mfcc_dataset, extract_spectrogram_dataset

feat_dir = os.path.join(tmp_root, "features")
X, y = extract_mfcc_dataset(manifest_csv, feature_dir=feat_dir, sr=SR)
print(f"\nMFCC: X.shape={X.shape}, y={y}")

# 4. Run spectrogram extraction
spec_dir = os.path.join(tmp_root, "spectrograms")
extract_spectrogram_dataset(manifest_csv, spec_dir=spec_dir, sr=SR)
for label in ("fox", "nonfox"):
    pngs = os.listdir(os.path.join(spec_dir, label))
    print(f"  {label}/: {len(pngs)} PNGs → {pngs}")

# 5. Re-run to verify caching (should skip all)
print("\n── Re-run (should hit cache) ──")
X2, y2 = extract_mfcc_dataset(manifest_csv, feature_dir=feat_dir, sr=SR)
extract_spectrogram_dataset(manifest_csv, spec_dir=spec_dir, sr=SR)

# Clean up
shutil.rmtree(tmp_root)
print("\n✅ features.py verified!")

---

## Step 5: `src/baseline_model.py`

Scikit-learn baseline classifiers with a `BaselineClassifier` wrapper class.

| Component | Description |
|---|---|
| `BaselineClassifier(model_type)` | Wraps `StandardScaler` + classifier in a sklearn `Pipeline`. Supports `'svm'`, `'random_forest'`, `'gradient_boosting'` |
| `.train(X, y)` | Fit pipeline, print training accuracy |
| `.evaluate(X, y)` | Return `{accuracy, precision, recall, f1, classification_report}`, save confusion-matrix heatmap |
| `.save(path)` / `.load(path)` | Pickle round-trip |
| `train_baseline(manifest, …)` | Load MFCCs → 70/15/15 split → train all 3 models → pick best by val F1 → evaluate on test → save |

**CLI:**
```bash
python -m src.baseline_model --manifest data/manifest.csv \
                             --feature_dir data/features/ \
                             --model_dir models/baseline/
```

In [ ]:
# Smoke-test BaselineClassifier on synthetic separable data
from src.baseline_model import BaselineClassifier
import tempfile, shutil

rng = np.random.RandomState(42)
n_per_class = 100
X_fox    = rng.randn(n_per_class, 80).astype(np.float32) + 2.0
X_nonfox = rng.randn(n_per_class, 80).astype(np.float32) - 2.0
X_syn = np.vstack([X_fox, X_nonfox])
y_syn = np.array([1]*n_per_class + [0]*n_per_class, dtype=np.int64)

# Shuffle
idx = rng.permutation(len(y_syn))
X_syn, y_syn = X_syn[idx], y_syn[idx]

# Split 70/30
split = int(0.7 * len(y_syn))
X_tr, X_te = X_syn[:split], X_syn[split:]
y_tr, y_te = y_syn[:split], y_syn[split:]

tmp_dir = tempfile.mkdtemp(prefix="bl_nb_test_")

for mt in ("svm", "random_forest", "gradient_boosting"):
    print(f"\n── {mt} ──")
    clf = BaselineClassifier(model_type=mt)
    clf.train(X_tr, y_tr)
    m = clf.evaluate(X_te, y_te,
                     save_cm_path=os.path.join(tmp_dir, f"cm_{mt}.png"))
    print(f"  Test → acc={m['accuracy']:.3f}  prec={m['precision']:.3f}  "
          f"rec={m['recall']:.3f}  f1={m['f1']:.3f}")

    # Save/load round-trip
    pkl = os.path.join(tmp_dir, f"{mt}.pkl")
    clf.save(pkl)
    clf2 = BaselineClassifier.load(pkl)
    assert np.array_equal(clf2.pipeline.predict(X_te), clf.pipeline.predict(X_te))
    print("  ✓ save/load round-trip OK")

shutil.rmtree(tmp_dir)
print("\n✅ baseline_model.py verified!")

---

## Step 6: `src/dataset.py`

PyTorch `Dataset` for spectrogram images with stratified splitting and augmentation.

| Component | Description |
|---|---|
| `FoxSpectrogramDataset` | Loads spectrogram PNGs, stratified 70/15/15 split, outputs `(3, H, W)` tensors |
| Augmentation (train) | Random horizontal flip, frequency masking (≤20 rows), time masking (≤20 cols), Gaussian noise (σ=0.01) |
| Normalisation | `[0,1]` → ImageNet mean/std (`[0.485,0.456,0.406]` / `[0.229,0.224,0.225]`) |
| `get_class_weights()` | Inverse-frequency weights for `nn.CrossEntropyLoss(weight=…)` |

In [ ]:
# Smoke-test FoxSpectrogramDataset with synthetic PNGs
import tempfile, shutil
import pandas as pd
from PIL import Image

tmp_ds = tempfile.mkdtemp(prefix="ds_nb_")
spec_dir_test = os.path.join(tmp_ds, "specs")
for lb in ("fox", "nonfox"):
    os.makedirs(os.path.join(spec_dir_test, lb))

rows = []
for lb, bv in [("fox", 200), ("nonfox", 50)]:
    for i in range(20):
        fid = f"{lb}_{i:04d}"
        arr = np.full((64, 64), bv + i, dtype=np.uint8)
        Image.fromarray(arr, mode="L").save(os.path.join(spec_dir_test, lb, f"{fid}.png"))
        rows.append({"file_id": fid, "source_file": f"{fid}.wav",
                      "clip_path": f"clips/{lb}/{fid}.wav", "label": lb,
                      "start_sec": 0.0, "end_sec": 3.0})
csv_test = os.path.join(tmp_ds, "manifest.csv")
pd.DataFrame(rows).to_csv(csv_test, index=False)

from src.dataset import FoxSpectrogramDataset

for split in ("train", "val", "test"):
    ds = FoxSpectrogramDataset(csv_test, spec_dir_test, split=split,
                               img_size=(64, 64), augment=(split == "train"))
    img, label = ds[0]
    print(f"{split:5s}: len={len(ds):3d}  shape={tuple(img.shape)}  "
          f"label={label}  range=[{img.min():.2f}, {img.max():.2f}]")

# Class weights
cw = FoxSpectrogramDataset.get_class_weights(
    FoxSpectrogramDataset(csv_test, spec_dir_test, split="train", img_size=(64,64)).labels
)
print(f"\nClass weights: {cw}")

shutil.rmtree(tmp_ds)
print("\n✅ dataset.py verified!")

---

## Step 7: `src/cnn_model.py` & `src/train_cnn.py`

### CNN Architecture (`FoxCNN`)

| Backbone | Source | Final head |
|---|---|---|
| `efficientnet_b0` | `torchvision.models` | `Dropout(p) → Linear(1280, 2)` |
| `resnet18` | `torchvision.models` | `Dropout(p) → Linear(512, 2)` |
| `mobilenet_v3_small` | `torchvision.models` | `Dropout(p) → Linear(1024, 2)` |

### Training Loop (`train_cnn`)

- **Data**: `FoxSpectrogramDataset` with augmentation on train split
- **Loss**: Weighted `CrossEntropyLoss` (inverse-frequency class weights)
- **Optimiser**: `AdamW` (weight_decay=1e-4)
- **Scheduler**: `CosineAnnealingLR`
- **Early stopping**: patience=7 epochs by val F1
- **Outputs**: `best.pt` checkpoint + `training_curves.png`

**CLI:**
```bash
python -m src.train_cnn --manifest data/manifest.csv --spec_dir data/spectrograms/ \
                        --backbone efficientnet_b0 --epochs 30 --batch_size 32
```

In [ ]:
# Test FoxCNN forward pass for all three backbones
import torch
from src.cnn_model import FoxCNN

dummy_input = torch.randn(2, 3, 128, 128)

for bb in ("efficientnet_b0", "resnet18", "mobilenet_v3_small"):
    model = FoxCNN(backbone=bb, pretrained=False, num_classes=2, dropout=0.3)
    out = model(dummy_input)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"✓ {bb:25s} → output {tuple(out.shape)}  params={n_params:,}")

print("\n✅ FoxCNN verified for all backbones!")

In [ ]:
# Quick 2-epoch training test on synthetic spectrograms
import tempfile, shutil
from PIL import Image
from src.train_cnn import train_cnn

tmp_cnn = tempfile.mkdtemp(prefix="cnn_nb_")
spec_dir_cnn = os.path.join(tmp_cnn, "specs")
for lb in ("fox", "nonfox"):
    os.makedirs(os.path.join(spec_dir_cnn, lb))

rows = []
for lb, bv in [("fox", 200), ("nonfox", 50)]:
    for i in range(16):
        fid = f"{lb}_{i:04d}"
        arr = np.random.randint(bv, bv + 30, (64, 64), dtype=np.uint8)
        Image.fromarray(arr, mode="L").save(os.path.join(spec_dir_cnn, lb, f"{fid}.png"))
        rows.append({"file_id": fid, "source_file": f"{fid}.wav",
                      "clip_path": f"clips/{lb}/{fid}.wav", "label": lb,
                      "start_sec": 0.0, "end_sec": 3.0})
csv_cnn = os.path.join(tmp_cnn, "manifest.csv")
pd.DataFrame(rows).to_csv(csv_cnn, index=False)

best = train_cnn(
    manifest_csv=csv_cnn, spec_dir=spec_dir_cnn,
    model_dir=os.path.join(tmp_cnn, "models"),
    backbone="resnet18", pretrained=False,
    epochs=2, batch_size=8, img_size=(64, 64), device="cpu",
)
ckpt = torch.load(best, map_location="cpu", weights_only=False)
print(f"\nCheckpoint: epoch={ckpt['epoch']}  val_f1={ckpt['val_f1']:.4f}")

shutil.rmtree(tmp_cnn)
print("✅ train_cnn verified!")

## Step 8 — `src/evaluate.py`

Unified evaluation module that works for **both** baseline (scikit-learn) and CNN (PyTorch) models.

### Public API

| Function | Purpose |
|---|---|
| `evaluate_model(model_type, model_path, manifest_csv, ...)` | Run inference on the **test** split, compute 7 metrics, save 3 diagnostic plots |
| `compare_models(baseline_metrics, cnn_metrics)` | Print a side-by-side pandas DataFrame comparison table |

### Metrics computed
`accuracy`, `precision_macro`, `recall_macro`, `f1_macro`, `f1_weighted`, `pr_auc`, `roc_auc`

### Plots saved
- **Confusion matrix** heatmap (`.png`)
- **Precision-Recall curve** with AP (`.png`)
- **ROC curve** with AUC (`.png`)

### CLI
```bash
python -m src.evaluate --model_type baseline --model_path models/baseline_best.pkl \
       --manifest data/manifest.csv --feature_dir data/features
python -m src.evaluate --model_type cnn --model_path models/cnn/best.pt \
       --manifest data/manifest.csv --spec_dir data/spectrograms
```

In [ ]:
# ── Step 8: Smoke-test src/evaluate.py ─────────────────────────────────
import os, sys, shutil, tempfile
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.join(PROJECT_ROOT, "fox_detection"))

# ── Build a tiny synthetic environment ───────────────────────────────
tmp = tempfile.mkdtemp(prefix="eval_test_")
print(f"[tmp dir] {tmp}")

rows = []
for i in range(10):
    rows.append({"clip_path": f"clips/fox_{i}.wav", "label": "fox"})
    rows.append({"clip_path": f"clips/nonfox_{i}.wav", "label": "nonfox"})
manifest_csv = os.path.join(tmp, "manifest.csv")
pd.DataFrame(rows).to_csv(manifest_csv, index=False)

# ── Monkey-patch inference to inject known outputs ───────────────────
import src.evaluate as ev

def _fake_baseline(*a, **kw):
    y_true = np.array([1, 0, 1, 0, 1, 0])
    y_pred = np.array([1, 0, 1, 1, 1, 0])
    y_prob = np.array([0.9, 0.1, 0.85, 0.6, 0.95, 0.2])
    return y_true, y_pred, y_prob

def _fake_cnn(*a, **kw):
    y_true = np.array([1, 0, 1, 0, 1, 0])
    y_pred = np.array([1, 0, 1, 0, 0, 0])
    y_prob = np.array([0.95, 0.05, 0.9, 0.1, 0.4, 0.15])
    return y_true, y_pred, y_prob

ev._infer_baseline = _fake_baseline
ev._infer_cnn      = _fake_cnn

out_dir = os.path.join(tmp, "eval_output")

# ── Evaluate both model types ────────────────────────────────────────
base_m = ev.evaluate_model("baseline", "dummy.pkl", manifest_csv,
                           feature_dir="x", output_dir=out_dir)
cnn_m  = ev.evaluate_model("cnn", "dummy.pt", manifest_csv,
                           spec_dir="x", output_dir=out_dir)

# ── Side-by-side comparison ──────────────────────────────────────────
df = ev.compare_models(base_m, cnn_m)

# ── Verify generated plot files ──────────────────────────────────────
for fn in ["confusion_matrix_baseline.png", "pr_curve_baseline.png",
           "roc_curve_baseline.png", "confusion_matrix_cnn.png",
           "pr_curve_cnn.png", "roc_curve_cnn.png"]:
    fp = os.path.join(out_dir, fn)
    assert os.path.isfile(fp), f"Missing: {fn}"
    print(f"  ✓ {fn}  ({os.path.getsize(fp):,} bytes)")

shutil.rmtree(tmp)
print("\n✅ evaluate.py smoke-test passed!")

## Step 9 — `src/demo.py` (Gradio Web App)

A Gradio-powered web interface for real-time fox vocalisation detection.

### Layout
| Widget | Description |
|---|---|
| **Audio input** | Upload `.wav` `.mp3` `.ogg` `.flac` or record via microphone |
| **Model selector** | Dropdown: *CNN (EfficientNet-B0)* / *Baseline SVM* |
| **Analyse button** | Triggers the full inference pipeline |

### Pipeline (on submit)
1. **Load & normalise** audio via `audio_utils.load_audio` + `normalise_audio`.
2. **Segment** into 3-second clips (`segment_fixed`, overlap=0.5 s).
3. **Per-clip inference** with the selected model (CNN → spectrogram → FoxCNN, or Baseline → MFCC → SVM).
4. **Aggregate**: label file as 🦊 FOX if > 30 % of clips are fox.
5. **Outputs**: prediction label + confidence + waveform with colour-coded clips + log-mel spectrogram of the most-confident fox clip.

### CLI
```bash
python src/demo.py --cnn_model models/cnn/best.pt \
                   --baseline_model models/baseline/model.pkl \
                   --manifest data/manifest.csv --port 7860
```

In [ ]:
# ── Step 9: Smoke-test src/demo.py (no server launch) ─────────────────
import os, sys, tempfile, shutil
import numpy as np
import soundfile as sf

sys.path.insert(0, os.path.join(PROJECT_ROOT, "fox_detection"))

# 1. Synthesise a 6-second test WAV
tmp = tempfile.mkdtemp(prefix="demo_test_")
t = np.linspace(0, 6.0, int(SR * 6.0), endpoint=False)
signal = (0.5 * np.sin(2 * np.pi * 440 * t)).astype(np.float32)
wav_path = os.path.join(tmp, "test.wav")
sf.write(wav_path, signal, SR)
print(f"[1] Test WAV: {wav_path}")

# 2. Fake baseline
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
import pickle

rng = np.random.RandomState(42)
X_tr = rng.randn(40, 80).astype(np.float32)
y_tr = np.array([0]*20 + [1]*20)
pipe = Pipeline([("scaler", StandardScaler()), ("clf", SVC(probability=True, random_state=42))])
pipe.fit(X_tr, y_tr)
pkl_path = os.path.join(tmp, "baseline.pkl")
with open(pkl_path, "wb") as f:
    pickle.dump({"model_type": "svm", "pipeline": pipe}, f)
print(f"[2] Fake baseline: {pkl_path}")

# 3. Fake CNN checkpoint
import torch
from src.cnn_model import FoxCNN
cnn = FoxCNN(backbone="efficientnet_b0", pretrained=False, num_classes=2)
pt_path = os.path.join(tmp, "best.pt")
torch.save({"backbone": "efficientnet_b0", "model_state_dict": cnn.state_dict()}, pt_path)
print(f"[3] Fake CNN: {pt_path}")

# 4. Load models & run predictions
from src.demo import load_models, predict, _MODEL_CACHE
load_models(cnn_path=pt_path, baseline_path=pkl_path)

for model_name in ["Baseline SVM", "CNN (EfficientNet-B0)"]:
    label, conf, wf_fig, sp_fig = predict(wav_path, model_name)
    print(f"\n--- {model_name} ---")
    print(f"  Prediction: {label}  |  Confidence: {conf}")
    import matplotlib.pyplot as plt
    plt.close(wf_fig); plt.close(sp_fig)
    print(f"  ✓ OK")

shutil.rmtree(tmp)
print("\n✅ demo.py smoke-test passed!")

## Step 10 – Analysis Notebooks

Five self-contained Jupyter notebooks live in `fox_detection/notebooks/`:

| # | Notebook | Purpose |
|---|----------|---------|
| 1 | `01_data_exploration.ipynb` | Load manifest, class-distribution bar chart, waveform + spectrogram plots, duration histogram, average spectrogram per class |
| 2 | `02_preprocessing.ipynb` | Demonstrate `load_audio → normalise → trim_silence` pipeline, before/after spectrograms, `segment_fixed` with all clips displayed |
| 3 | `03_baseline_model.ipynb` | Extract/load MFCCs, 70/15/15 split, train SVM / Random-Forest / Gradient-Boosting, confusion matrices, F1 bar chart |
| 4 | `04_cnn_model.ipynb` | Display augmented images, train CNN with `train_cnn()`, learning-curve plots, per-class metrics from saved checkpoint |
| 5 | `05_evaluation.ipynb` | Evaluate both model families, `compare_models()` summary table, combined PR + ROC curves on one figure |

All notebooks include **try / except** guards so they fall back to synthetic data when real audio or model artefacts are missing.

In [26]:
# ── Step 10: Verify notebooks exist and are valid ──────────────────────
import json, pathlib

nb_dir = pathlib.Path(PROJECT_ROOT) / "notebooks"
expected = [
    "01_data_exploration.ipynb",
    "02_preprocessing.ipynb",
    "03_baseline_model.ipynb",
    "04_cnn_model.ipynb",
    "05_evaluation.ipynb",
]

for name in expected:
    p = nb_dir / name
    assert p.exists(), f"Missing: {p}"
    nb = json.loads(p.read_text())
    n_cells = len(nb["cells"])
    print(f"✓ {name:35s}  {n_cells:>3} cells  nbformat {nb['nbformat']}")

print("\n✅ All 5 analysis notebooks present and valid.")

✓ 01_data_exploration.ipynb             10 cells  nbformat 4
✓ 02_preprocessing.ipynb                10 cells  nbformat 4
✓ 03_baseline_model.ipynb               12 cells  nbformat 4
✓ 04_cnn_model.ipynb                    12 cells  nbformat 4
✓ 05_evaluation.ipynb                   12 cells  nbformat 4

✅ All 5 analysis notebooks present and valid.


## Step 11 – Pipeline Script & Unit Tests

### `run_pipeline.sh`
A single `bash` script that executes the full pipeline end-to-end (segment → extract features → train baseline → train CNN → evaluate both → launch demo). Run from the `fox_detection/` directory:
```bash
cd fox_detection && bash run_pipeline.sh
```

### `tests/test_audio_utils.py`
17 pytest tests covering four core components:

| Class | What is tested |
|-------|----------------|
| `TestLoadAudio` | correct SR, float32 dtype, mono 1-D, sample count, resampling |
| `TestNormaliseAudio` | output ∈ [-1, 1], peak == 1, silent signal, dtype preservation |
| `TestComputeMfccFeatures` | shape (80,), custom n_mfcc, float32, no NaNs |
| `TestSegmentFixed` | exact clip length, overlap handling, clip count, zero-padding |

In [27]:
# ── Step 11: Verify run_pipeline.sh & run pytest ───────────────────────
import subprocess, pathlib

project = pathlib.Path(PROJECT_ROOT)

# 11a – run_pipeline.sh exists and is executable
sh = project / "run_pipeline.sh"
assert sh.exists(), f"Missing: {sh}"
print(f"✓ {sh.name} exists ({sh.stat().st_size} bytes)")

# 11b – tests/test_audio_utils.py exists
test_file = project / "tests" / "test_audio_utils.py"
assert test_file.exists(), f"Missing: {test_file}"
print(f"✓ {test_file.relative_to(project)} exists")

# 11c – run pytest
result = subprocess.run(
    ["python", "-m", "pytest", str(test_file), "-v", "--tb=short"],
    capture_output=True, text=True, cwd=str(project),
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
assert result.returncode == 0, "pytest failed!"
print("✅ All 17 tests passed.")

✓ run_pipeline.sh exists (1229 bytes)
✓ tests/test_audio_utils.py exists
============================= test session starts ==============================
platform darwin -- Python 3.14.1, pytest-9.0.2, pluggy-1.6.0 -- /Users/dyx/Documents/TECHIN513A/Final/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/dyx/Documents/TECHIN513A/Final/fox_detection
plugins: anyio-4.12.1
collecting ... collected 17 items

tests/test_audio_utils.py::TestLoadAudio::test_returns_correct_sr PASSED [  5%]
tests/test_audio_utils.py::TestLoadAudio::test_returns_float32 PASSED    [ 11%]
tests/test_audio_utils.py::TestLoadAudio::test_mono_1d PASSED            [ 17%]
tests/test_audio_utils.py::TestLoadAudio::test_sample_count_roughly_correct PASSED [ 23%]
tests/test_audio_utils.py::TestLoadAudio::test_resample PASSED           [ 29%]
tests/test_audio_utils.py::TestNormaliseAudio::test_output_in_range PASSED [ 35%]
tests/test_audio_utils.py::TestNormaliseAudio::test_peak_is_one PASSED   [ 41%]
tests/test_au